In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
movies=pd.read_csv(
    '../ml-100k/u.item',
    sep="|",
    encoding="latin-1",
    header=None
)

In [6]:
movies=movies[[0,1]+list(range(5,24))]
movies.columns=[
    "movie_id","title",
    "unknown","Action","Adventure","Animation","Children","Comedy","Crime",
    "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
    "Romance","Sci-Fi","Thriller","War","Western"
]

In [8]:
genre_cols = [
    'unknown','Action','Adventure','Animation',"Children",
    'Comedy','Crime','Documentary','Drama','Fantasy',
    'Film-Noir','Horror','Musical','Mystery',
    'Romance','Sci-Fi','Thriller','War','Western'
]

movies[genre_cols].head()

,unknown,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0


## Movie features

In [34]:
movies["num_genres"] = movies[genre_cols].sum(axis=1)

### genre Text

In [13]:
movies = movies.copy()

movies['genres_text'] = movies.apply(
    lambda row: " ".join(
        [genre for genre in genre_cols if row[genre] == 1]
    ),
    axis=1
)

In [15]:
movies[["title","genres_text"]].head()

,title,genres_text
0,Toy Story (1995),Animation Children Comedy
1,GoldenEye (1995),Action Adventure Thriller
2,Four Rooms (1995),Thriller
3,Get Shorty (1995),Action Comedy Drama
4,Copycat (1995),Crime Drama Thriller


In [40]:
movies["release_year"] = (
    movies["title"]
    .str.extract(r"\((\d{4})\)")
    .astype(float)
)

next step :- TF-IDF se vector nikalenge or unke bich similarity dekhenge

### user features

now same age group may ahve same interest--> lets group them

In [17]:
users=pd.read_csv(
    '../ml-100k/u.user',
    sep="|",
    names=['user_id','age','gender','occupation','zip_code']
)

In [19]:
bins=[0,18,25,35,50,100]

labels=["Teen","Young Adult","Adult","Middle Age","Senior"]

users['age_group']=pd.cut(
    users['age'],
    bins=bins,
    labels=labels
)

In [20]:
users[["age", "age_group"]].head()

,age,age_group
0,24,Young Adult
1,53,Senior
2,23,Young Adult
3,24,Young Adult
4,33,Adult


In [ ]:
#occupation encoding

In [38]:
occupation_encoded=pd.get_dummies( #perform one hot encoding
    users['occupation'],
    prefix='occ',
    dtype=int
)

In [39]:
#combine

users_features=pd.concat([users,occupation_encoded],axis=1)

In [25]:
ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

In [ ]:
# user activity
#the user who have rated more movies can be trusted more
user_activity=ratings.groupby(
    'user_id'
)['rating'].count()

users['rating_count']=users['user_id'].map(
    user_activity
)

In [36]:
#rating variance by user

# the user who have given more varied rating can be trusted more because he likes distinguis movies

user_rating_std=ratings.groupby(
    'user_id'
)['rating'].std()

users['rating_std']=users['user_id'].map(
    user_rating_std
)

### Rating features

In [26]:
#finding user mean_ratings

user_mean=ratings.groupby( 
    'user_id'
)['rating'].mean()

now we know ratings of movie which is rated by more number of people can be trusted more which is rated by less number of people

In [27]:
movie_mean=ratings.groupby(
    'movie_id'
)['rating'].mean()

movie_count=ratings.groupby(
    'movie_id'
)['rating'].count()

In [31]:
#normalise ratings

ratings['user_mean']=ratings.groupby(
    'user_id'
)['rating'].transform('mean')

ratings['normalised_rating']=(
    ratings['rating']-ratings['user_mean']
)

In [33]:
ratings[[
    "user_id",
    "movie_id",
    "rating",
    "user_mean",
    "normalised_rating"
]].head(10)

,user_id,movie_id,rating,user_mean,normalised_rating
0,196,242,3,3.615385,-0.615385
1,186,302,3,3.413043,-0.413043
2,22,377,1,3.351562,-2.351562
3,244,51,2,3.651261,-1.651261
4,166,346,1,3.550000,-2.550000
5,298,474,4,4.031496,-0.031496
6,115,265,2,3.934783,-1.934783
7,253,465,5,3.979381,1.020619
8,305,451,3,3.409910,-0.409910
9,6,86,3,3.635071,-0.635071


### popularity score

In [28]:
movie_stats = ratings.groupby(
    "movie_id"
).agg(
    avg_rating=("rating","mean"),
    rating_count=("rating","count")
).reset_index()

In [37]:
#weighted probabiilty

movie_stats['popularity_score']=(
    movie_stats['avg_rating']*np.log1p(movie_stats['rating_count'])
)

### Interaction matrix

In [29]:
interaction_matrix=ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

In [41]:
interaction_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Movie Features**

- genres_text
- num_genres
- release_year

**User Features**

- age_group
- occupation
- rating_count -->no. of rating given by usr
- rating_std

**Rating Features**

- user mean --avg rating given by user
- normalised_rating
- 

**movie_statistics_features**

- avg_rating
- rating_count
- popularity score


**derived structure**

interaction matrix

## Saving features to use later

In [42]:
movies.to_parquet(
    "../data/processed/movies_features.parquet",
    index=False
)

In [43]:
users_features.to_parquet(
    "../data/processed/users_features.parquet",
    index=False
)

In [44]:
ratings.to_parquet(
    "../data/processed/ratings_features.parquet",
    index=False
)

In [45]:
movie_stats.to_parquet(
    "../data/processed/movie_stats.parquet",
    index=False
)

In [46]:
interaction_matrix.to_pickle(
    "../data/processed/interaction_matrix.pkl"
)